In [2]:
df = spark.read.parquet("Files/food-nutrition-health-data/food.parquet")
df.printSchema()

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 7, Finished, Available, Finished, False)

root
 |-- additives_n: integer (nullable = true)
 |-- additives_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- allergens_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- brands_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- brands: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- categories_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories_properties: struct (nullable = true)
 |    |-- ciqual_food_code: integer (nullable = true)
 |    |-- agribalyse_food_code: integer (nullable = true)
 |    |-- agribalyse_proxy_food_code: integer (nullable = true)
 |-- checkers_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- ciqual_food_name_tags: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- cities_tags: array (nullable = true)
 |    |-- element: string (containsNul

In [3]:
df = spark.read.csv("Files/food-environmental-impact/ghg-per-kg-poore.csv", header=True)
df.show(5)

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 8, Finished, Available, Finished, False)

+-----------------+----+-------------------------------------+
|           Entity|Year|Greenhouse gas emissions per kilogram|
+-----------------+----+-------------------------------------+
|           Apples|2010|                                 0.43|
|          Bananas|2010|                                 0.86|
|           Barley|2010|                                 1.18|
| Beef (beef herd)|2010|                                99.48|
|Beef (dairy herd)|2010|                                 33.3|
+-----------------+----+-------------------------------------+
only showing top 5 rows



In [4]:
df_owid = spark.read.csv("Files/food-environmental-impact/ghg-per-kg-poore.csv", header=True)
entities = df_owid.select("Entity").distinct().orderBy("Entity")
entities.show(50, truncate=False)

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 9, Finished, Available, Finished, False)

+-----------------+
|Entity           |
+-----------------+
|Apples           |
|Bananas          |
|Barley           |
|Beef (beef herd) |
|Beef (dairy herd)|
|Beet Sugar       |
|Berries & Grapes |
|Brassicas        |
|Cane Sugar       |
|Cassava          |
|Cheese           |
|Citrus Fruit     |
|Coffee           |
|Dark Chocolate   |
|Eggs             |
|Fish (farmed)    |
|Groundnuts       |
|Lamb & Mutton    |
|Maize            |
|Milk             |
|Nuts             |
|Oatmeal          |
|Onions & Leeks   |
|Other Fruit      |
|Other Pulses     |
|Other Vegetables |
|Peas             |
|Pig Meat         |
|Potatoes         |
|Poultry Meat     |
|Prawns (farmed)  |
|Rice             |
|Root Vegetables  |
|Soy milk         |
|Tofu             |
|Tomatoes         |
|Wheat & Rye      |
|Wine             |
+-----------------+



In [5]:
df_off = spark.read.parquet("Files/food-nutrition-health-data/food.parquet")

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 10, Finished, Available, Finished, False)

In [6]:
for col in df_off.columns:
    if "categor" in col.lower() or "label" in col.lower():
        print(col)

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 11, Finished, Available, Finished, False)

categories
categories_tags
categories_properties
compared_to_category
labels_tags
labels


In [7]:
df_off.select("product_name", "categories_tags", "labels_tags").sample(0.0005).show(40, truncate=False)

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 12, Finished, Available, Finished, False)

+----------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------------------------------+
|product_name                                                                                                                                  |categories_tags                                                                                                                                                                                                                                                           |labels_tags                          |
+---------------------------------------------------------------------------------------------------

In [8]:
display(df_off.select("product_name", "categories_tags", "labels_tags").sample(0.0005).limit(40))

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, baec42c7-4216-47e3-9e59-feaa18e3583c)

In [9]:
from pyspark.sql.functions import col, size, array_contains

total = df_off.count()

# Rows where categories_tags is null, empty, or only contains placeholder junk values
useless_categories = df_off.filter(
    col("categories_tags").isNull() |
    (size(col("categories_tags")) == 0) |
    array_contains(col("categories_tags"), "en:undefined") |
    array_contains(col("categories_tags"), "en:null")
).count()

usable = total - useless_categories

print(f"Total products: {total}")
print(f"Usable categories_tags: {usable} ({round(usable/total*100, 1)}%)")
print(f"Unusable categories_tags: {useless_categories} ({round(useless_categories/total*100, 1)}%)")

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 14, Finished, Available, Finished, False)

Total products: 4655195
Usable categories_tags: 1836653 (39.5%)
Unusable categories_tags: 2818542 (60.5%)


In [11]:
from pyspark.sql.functions import col, size, array_contains

untagged = df_off.filter(
    col("categories_tags").isNull() |
    (size(col("categories_tags")) == 0) |
    array_contains(col("categories_tags"), "en:undefined") |
    array_contains(col("categories_tags"), "en:null")
)

tagged = df_off.filter(
    col("categories_tags").isNotNull() &
    (size(col("categories_tags")) > 0) &
    (~array_contains(col("categories_tags"), "en:undefined")) &
    (~array_contains(col("categories_tags"), "en:null"))
)

print("Untagged group — top countries:")
untagged.groupBy("countries_tags").count().orderBy(col("count").desc()).show(10, truncate=False)

print("Tagged group — top countries:")
tagged.groupBy("countries_tags").count().orderBy(col("count").desc()).show(10, truncate=False)

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 21, Finished, Available, Finished, False)

Untagged group — top countries:
+-------------------+------+
|countries_tags     |count |
+-------------------+------+
|[en:france]        |614917|
|[en:united-states] |527976|
|[en:spain]         |221658|
|[en:germany]       |216631|
|[en:italy]         |138998|
|[en:united-kingdom]|115258|
|[en:canada]        |90652 |
|[en:netherlands]   |61312 |
|[en:ireland]       |59375 |
|[en:australia]     |58055 |
+-------------------+------+
only showing top 10 rows

Tagged group — top countries:
+-------------------+------+
|countries_tags     |count |
+-------------------+------+
|[en:france]        |520697|
|[en:united-states] |373066|
|[en:germany]       |126081|
|[en:spain]         |109128|
|[en:italy]         |107690|
|[en:united-kingdom]|60210 |
|[en:netherlands]   |39075 |
|[en:belgium]       |38107 |
|[en:switzerland]   |35423 |
|[en:canada]        |25909 |
+-------------------+------+
only showing top 10 rows



In [12]:
from pyspark.sql.functions import expr

total = 4655195

coverage = df_off.agg(
    expr("sum(case when lower(nutriscore_grade) in ('a','b','c','d','e') then 1 else 0 end)").alias("has_nutriscore"),
    expr("sum(case when nova_group between 1 and 4 then 1 else 0 end)").alias("has_nova"),
    expr("sum(case when size(nutriments) > 0 then 1 else 0 end)").alias("has_nutriments"),
    expr("sum(case when size(food_groups_tags) > 0 then 1 else 0 end)").alias("has_food_groups"),
    expr("sum(case when size(ingredients_analysis_tags) > 0 then 1 else 0 end)").alias("has_ingredients_analysis"),
).collect()[0].asDict()

for k, v in coverage.items():
    print(f"{k}: {v} ({round(v/total*100, 1)}%)")

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 22, Finished, Available, Finished, False)

has_nutriscore: 1387705 (29.8%)
has_nova: 1144186 (24.6%)
has_nutriments: 3588812 (77.1%)
has_food_groups: 1465891 (31.5%)
has_ingredients_analysis: 1377719 (29.6%)


In [13]:
from pyspark.sql.functions import explode, col

sample = df_off.sample(0.01)

print("ingredients_analysis_tags values:")
sample.select(explode("ingredients_analysis_tags").alias("tag")).groupBy("tag").count().orderBy(col("count").desc()).show(15, truncate=False)

print("nutriments names:")
sample.select(explode("nutriments").alias("n")).select("n.name").groupBy("name").count().orderBy(col("count").desc()).show(30, truncate=False)

StatementMeta(, 9c33cb3f-0228-4f9c-9af9-d54b5104cefb, 23, Finished, Available, Finished, False)

ingredients_analysis_tags values:
+----------------------------+-----+
|tag                         |count|
+----------------------------+-----+
|en:palm-oil-free            |8005 |
|en:vegetarian-status-unknown|6730 |
|en:non-vegan                |5534 |
|en:vegan-status-unknown     |4444 |
|en:vegetarian               |4149 |
|en:vegan                    |3209 |
|en:palm-oil-content-unknown |2916 |
|en:non-vegetarian           |1907 |
|en:maybe-vegetarian         |1289 |
|en:may-contain-palm-oil     |1150 |
|en:palm-oil                 |1130 |
|en:maybe-vegan              |815  |
+----------------------------+-----+

nutriments names:
+---------------------------------------------------+-----+
|name                                               |count|
+---------------------------------------------------+-----+
|energy                                             |33908|
|energy-kcal                                        |33821|
|carbohydrates                                      |33